# IVF Insight — Phase 5: Robustness, Sensitivity & Model Validation

**Author:** Senior Biomedical Data Scientist & Research Engineer  
**Authoritative Source:** `data/raw/Dataset Clean.xlsx` (5,000 candidate cycles; rows 0–4999)  
**Scope:** Validation audit of Phase 4 predictive models across temporal drift, cutoff sensitivity, calibration, feature stability, extreme observations, and decision thresholds.  

> [!IMPORTANT]
> **Clinical Non-Claim Notice:** Assisted reproductive technology (ART) involves complex sequential clinical decisions. This predictive modeling study is an educational portfolio project. It does **not** constitute a clinical decision support tool, medical device, or basis for clinical management. Predictive importance must **never** be interpreted as causal biological efficacy.

## 1. Environment Setup & Validation Imports

Import analytical components from `ivf_insight.validation`, `ivf_insight.eda`, and `ivf_insight.experiments`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import Image, display

from ivf_insight.eda import load_clean_cohorts
from ivf_insight.validation import (
    analyze_temporal_drift,
    evaluate_cutoff_sensitivity,
    compute_logistic_regression_odds_ratios,
    compute_calibration_diagnostics,
    evaluate_feature_importance_stability,
    evaluate_sensitivity_analyses,
    evaluate_clinical_thresholds,
)

REPORTS_DIR = Path('../reports')
ROBUST_FIG_DIR = REPORTS_DIR / 'robustness_figures'
print('Loaded validation module. Reports directory:', REPORTS_DIR.resolve())


## 2. Temporal Drift & Prevalence Investigation

We analyze annual cycle counts, outcome prevalence, cycle non-return rates, and feature distribution shifts across historical (< 2024-01-01) and prospective (>= 2024-01-01) periods.

In [ ]:
df_yearly = pd.read_csv(REPORTS_DIR / 'temporal_drift_yearly.csv')
display(df_yearly)

if (ROBUST_FIG_DIR / 'temporal_drift_prevalence.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'temporal_drift_prevalence.png'), width=900))


### Feature Distribution Shifts (Kolmogorov-Smirnov & Population Stability Index)

Comparing pre-2024 vs post-2024 feature distributions in the Embryo-Transfer Cohort.

In [ ]:
df_drift = pd.read_csv(REPORTS_DIR / 'temporal_drift.csv')
tr_drift = df_drift[df_drift['Cohort'] == 'Embryo-Transfer Cohort'].sort_values('Standardized_Mean_Diff')
display(tr_drift[['Feature', 'Historical_Mean', 'Prospective_Mean', 'Standardized_Mean_Diff', 'KS_Statistic', 'KS_P_Value', 'Population_Stability_Index', 'Shift_Severity']])

if (ROBUST_FIG_DIR / 'feature_drift_ks_smd.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'feature_drift_ks_smd.png'), width=800))


## 3. Chronological Cutoff Sensitivity Analysis

Testing whether model performance is an artifact of the specific `2024-01-01` boundary by validating across `2023-01-01`, `2024-01-01`, and `2024-07-01`.

In [ ]:
df_cutoff = pd.read_csv(REPORTS_DIR / 'cutoff_sensitivity.csv')
tr_scen_c = df_cutoff[(df_cutoff['Cohort'] == 'Embryo-Transfer Cohort') & (df_cutoff['Scenario'] == 'Scenario C (Pre-Transfer)')]
display(tr_scen_c[['Cutoff_Date', 'Model', 'Train_N', 'Train_Pos_Rate', 'Holdout_N', 'Holdout_Pos_Rate', 'ROC_AUC', 'Balanced_Accuracy', 'Brier_Score']])

if (ROBUST_FIG_DIR / 'cutoff_sensitivity_comparison.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'cutoff_sensitivity_comparison.png'), width=700))


## 4. Statistically Rigorous Logistic Regression Odds Ratios

Estimating standardized odds ratios ($OR = e^\beta$) and asymptotic 95% confidence intervals via `statsmodels.Logit` on standardized training data.

In [ ]:
df_or = pd.read_csv(REPORTS_DIR / 'logistic_regression_odds_ratios.csv')
or_tr_c = df_or[(df_or['Cohort'] == 'Embryo-Transfer Cohort') & (df_or['Scenario'] == 'Scenario C (Pre-Transfer)')]
display(or_tr_c[['Feature', 'Coefficient_Beta', 'Standard_Error', 'Z_Statistic', 'P_Value', 'Odds_Ratio', 'OR_CI_95_Formatted']])


## 5. Calibration Diagnostics & Recalibration Analysis

Assessing whether models are systematically overconfident or underconfident via Harrell/Cox logistic recalibration: $\text{logit}(Y) = \alpha + \beta \cdot \text{logit}(p)$.

In [ ]:
df_calib = pd.read_csv(REPORTS_DIR / 'calibration_diagnostics.csv')
display(df_calib[['Cohort', 'Scenario', 'Model', 'Brier_Score', 'Calibration_Intercept', 'Calibration_Slope', 'Systematic_Bias', 'Spread_Assessment']])

if (ROBUST_FIG_DIR / 'calibration_diagnostics_reliability.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'calibration_diagnostics_reliability.png'), width=900))


## 6. Feature Importance Stability & Sensitivity Analyses

Bootstrap resampling ($B=100$) of historical development data to quantify Gini importance stability and rank confidence intervals.

In [ ]:
df_stab = pd.read_csv(REPORTS_DIR / 'feature_stability.csv')
display(df_stab[['Feature', 'Mean_Gini_Importance', 'Std_Gini_Importance', 'Gini_CI_95_Low', 'Gini_CI_95_High', 'Mean_Rank', 'Rank_IQR']])

if (ROBUST_FIG_DIR / 'feature_importance_stability_bootstrap.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'feature_importance_stability_bootstrap.png'), width=800))


### Sensitivity Analysis: Extreme AMH Observations ($482.1$) and Leave-One-Feature-Out (LOFO) Ablation

Assessing whether Scenario C is fragile to extreme values or overly reliant on any single variable.

In [ ]:
df_sens = pd.read_csv(REPORTS_DIR / 'sensitivity_analyses.csv')
display(df_sens[['Sensitivity_Domain', 'Specification', 'Features_Count', 'ROC_AUC', 'PR_AUC', 'Balanced_Accuracy', 'Brier_Score']])

if (ROBUST_FIG_DIR / 'feature_ablation_sensitivity.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'feature_ablation_sensitivity.png'), width=800))


## 7. Clinical Decision Threshold Trade-offs & Prevalence Effects

Evaluating Sensitivity, Specificity, PPV, NPV, and Balanced Accuracy across probability cutoffs, illustrating Bayes' theorem in a high-prevalence holdout.

In [ ]:
df_thresh = pd.read_csv(REPORTS_DIR / 'threshold_analysis.csv')
display(df_thresh[['Probability_Threshold', 'Sensitivity', 'Specificity', 'Positive_Predictive_Value_PPV', 'Negative_Predictive_Value_NPV', 'Balanced_Accuracy', 'Overall_Accuracy', 'F1_Score']])

if (ROBUST_FIG_DIR / 'clinical_threshold_tradeoffs.png').exists():
    display(Image(filename=str(ROBUST_FIG_DIR / 'clinical_threshold_tradeoffs.png'), width=800))


## 8. Summary of Methodological Safeguards & Research Constraints

1. **No External Validation:** Findings apply strictly to internal chronological holdout partitions of this single-center dataset.
2. **Prevalence Drift Awareness:** Shift to 78.7% success in prospective transfer cycles necessitates recalibration before probability interpretation.
3. **Missing Patient Linkage:** Repeated treatment cycles from identical couples across time cannot be fully isolated without master patient IDs.
4. **No Clinical Claims:** This educational analysis provides no basis for changing clinical treatment protocols or advising patients.
